# 04. Preparación de datos para PCA y aprendizaje automático

Este cuaderno parte de la base física `geminidas_v1`, separa 2019–2024 de 2025 y aprende con entrenamiento la referencia física y las transformaciones que pueden aplicarse después al año reservado de prueba. El resultado son las bases `geminidas_modelado_v2` utilizadas por los notebooks de PCA, modelado y evaluación.


In [ ]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd


## 1. Carga y contrato de entrada


In [ ]:
# Localizar la raíz del proyecto

ruta_actual = Path.cwd().resolve()

raiz_proyecto = next(
    (
        ruta
        for ruta in [ruta_actual, *ruta_actual.parents]
        if (ruta / "pyproject.toml").is_file()
    ),
    None,
)

if raiz_proyecto is None:
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Abre VS Code o Jupyter desde el repositorio de la investigación."
    )

ruta_base_fisica = (
    raiz_proyecto
    / "data"
    / "processed"
    / "geminidas_v1"
    / "geminidas_estandar_analisis_v1.parquet"
)

ruta_diccionario_fisico = (
    raiz_proyecto
    / "data"
    / "processed"
    / "geminidas_v1"
    / "metadata"
    / "diccionario_variables_geminidas_v1.csv"
)

carpeta_modelado = (
    raiz_proyecto
    / "data"
    / "processed"
    / "geminidas_modelado_v2"
)

for ruta in [ruta_base_fisica, ruta_diccionario_fisico]:
    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encontró {ruta.relative_to(raiz_proyecto)}. "
            "Ejecuta primero el notebook 03."
        )

print("Raíz del proyecto:", raiz_proyecto)
print("Base física:", ruta_base_fisica.relative_to(raiz_proyecto))
print("Diccionario:", ruta_diccionario_fisico.relative_to(raiz_proyecto))


In [ ]:
# Cargar la base física y comprobar las variables necesarias

df_base_modelado = pd.read_parquet(ruta_base_fisica)
df_diccionario_fisico = pd.read_csv(ruta_diccionario_fisico)

columnas_necesarias = [
    "id_fila_origen",
    "Beginning (UTC Time)",
    "source_year",
    "nivel_seleccion",
    "delta_sol_lon_deg",
    "lambda_sce_deg",
    "BETgeo (deg)",
    "Vgeo (km/s)",
    "magnitud_analisis",
    "fotometria_utilizable",
    "flag_fotometria_sospechosa",
    "pertenece_muestra_estricta",
    "Qc (deg)",
    "MedianFitErr (arcsec)",
    "Participating (stations)",
    "sigma_radiante_aprox",
    "Vgeo_sigma",
    "flag_vgeo_sigma_cero",
]

columnas_faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in df_base_modelado.columns
]

if columnas_faltantes:
    raise KeyError(
        f"Faltan variables necesarias en geminidas_v1: {columnas_faltantes}"
    )

if not df_base_modelado["id_fila_origen"].is_unique:
    raise ValueError("La base física contiene identificadores repetidos.")

anios_observados = set(df_base_modelado["source_year"].unique())
anios_esperados = set(range(2019, 2026))

if anios_observados != anios_esperados:
    raise ValueError(
        "Los años de la base no coinciden con 2019–2025: "
        f"{sorted(anios_observados)}"
    )

if "variable" not in df_diccionario_fisico.columns:
    raise KeyError("El diccionario v1 no contiene la columna 'variable'.")

print("Dimensión de la base física:", df_base_modelado.shape)
print("Años presentes:", sorted(anios_observados))


## 2. Partición temporal


In [ ]:
# Separar 2019–2024 como desarrollo y reservar 2025 para evaluación posterior

df_modelado = df_base_modelado.copy()

df_modelado["particion_modelado"] = np.where(
    df_modelado["source_year"] <= 2024,
    "entrenamiento",
    "prueba",
)

df_entrenamiento = df_modelado.loc[
    df_modelado["particion_modelado"] == "entrenamiento"
].copy()

df_prueba = df_modelado.loc[
    df_modelado["particion_modelado"] == "prueba"
].copy()

ids_entrenamiento = set(df_entrenamiento["id_fila_origen"])
ids_prueba = set(df_prueba["id_fila_origen"])

if ids_entrenamiento & ids_prueba:
    raise ValueError("Entrenamiento y prueba comparten identificadores.")

if set(df_entrenamiento["source_year"].unique()) != set(range(2019, 2025)):
    raise ValueError("El entrenamiento no contiene exactamente 2019–2024.")

if set(df_prueba["source_year"].unique()) != {2025}:
    raise ValueError("La prueba no contiene exclusivamente 2025.")

resumen_particion_fisica = pd.DataFrame(
    [
        {
            "particion": "entrenamiento",
            "anios": "2019–2024",
            "trayectorias": len(df_entrenamiento),
            "trayectorias_estrictas": int(
                df_entrenamiento["pertenece_muestra_estricta"].sum()
            ),
        },
        {
            "particion": "prueba",
            "anios": "2025",
            "trayectorias": len(df_prueba),
            "trayectorias_estrictas": int(
                df_prueba["pertenece_muestra_estricta"].sum()
            ),
        },
    ]
)

display(resumen_particion_fisica)


## 3. Referencia física aprendida con 2019–2024


In [ ]:
# Ajustar los modelos de deriva exclusivamente con entrenamiento

def ajustar_polinomio(x, y, grado):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    coeficientes = np.polynomial.polynomial.polyfit(x, y, grado)
    valores_ajustados = np.polynomial.polynomial.polyval(x, coeficientes)
    residuos = y - valores_ajustados

    suma_cuadrados_residuos = np.sum(residuos ** 2)
    suma_cuadrados_total = np.sum((y - np.mean(y)) ** 2)

    r_cuadrada = 1 - suma_cuadrados_residuos / suma_cuadrados_total
    rmse = np.sqrt(np.mean(residuos ** 2))

    return coeficientes, r_cuadrada, rmse


columnas_ajuste_centro = [
    "delta_sol_lon_deg",
    "lambda_sce_deg",
    "BETgeo (deg)",
    "Vgeo (km/s)",
]

valores_centro = df_entrenamiento[columnas_ajuste_centro].to_numpy(dtype=float)

if not np.isfinite(valores_centro).all():
    raise ValueError(
        "El entrenamiento contiene valores no finitos en las variables del centro físico."
    )

configuracion_centro = {
    "lambda_sce": {"columna": "lambda_sce_deg", "grado": 1},
    "beta_geo": {"columna": "BETgeo (deg)", "grado": 2},
    "vgeo": {"columna": "Vgeo (km/s)", "grado": 1},
}

x_entrenamiento = df_entrenamiento["delta_sol_lon_deg"].to_numpy(dtype=float)

ajustes_centro_train = {}
filas_parametros_centro = []

for variable, configuracion in configuracion_centro.items():
    y_entrenamiento = df_entrenamiento[
        configuracion["columna"]
    ].to_numpy(dtype=float)

    coeficientes, r_cuadrada, rmse = ajustar_polinomio(
        x_entrenamiento,
        y_entrenamiento,
        configuracion["grado"],
    )

    ajustes_centro_train[variable] = {
        "columna": configuracion["columna"],
        "grado": configuracion["grado"],
        "coeficientes": coeficientes,
    }

    filas_parametros_centro.append(
        {
            "variable": variable,
            "modelo": (
                "Cuadrático"
                if configuracion["grado"] == 2
                else "Lineal"
            ),
            "trayectorias_entrenamiento": len(y_entrenamiento),
            "intercepto": coeficientes[0],
            "coeficiente_lineal": coeficientes[1],
            "coeficiente_cuadratico": (
                coeficientes[2]
                if configuracion["grado"] == 2
                else 0.0
            ),
            "r_cuadrada": r_cuadrada,
            "rmse": rmse,
        }
    )

df_parametros_centro_train = pd.DataFrame(filas_parametros_centro)

display(df_parametros_centro_train)


In [ ]:
# Aplicar la misma referencia de entrenamiento a ambas particiones

df_modelado_trainseguro = df_modelado.copy()

x_todos = df_modelado_trainseguro[
    "delta_sol_lon_deg"
].to_numpy(dtype=float)

df_modelado_trainseguro["lambda_sce_esperada_train_deg"] = (
    np.polynomial.polynomial.polyval(
        x_todos,
        ajustes_centro_train["lambda_sce"]["coeficientes"],
    )
)

df_modelado_trainseguro["beta_esperada_train_deg"] = (
    np.polynomial.polynomial.polyval(
        x_todos,
        ajustes_centro_train["beta_geo"]["coeficientes"],
    )
)

df_modelado_trainseguro["vgeo_esperada_train_km_s"] = (
    np.polynomial.polynomial.polyval(
        x_todos,
        ajustes_centro_train["vgeo"]["coeficientes"],
    )
)

df_modelado_trainseguro["residuo_lambda_train_deg"] = (
    (
        df_modelado_trainseguro["lambda_sce_deg"]
        - df_modelado_trainseguro["lambda_sce_esperada_train_deg"]
        + 180
    )
    % 360
    - 180
)

df_modelado_trainseguro["residuo_beta_train_deg"] = (
    df_modelado_trainseguro["BETgeo (deg)"]
    - df_modelado_trainseguro["beta_esperada_train_deg"]
)

df_modelado_trainseguro["residuo_vgeo_train_km_s"] = (
    df_modelado_trainseguro["Vgeo (km/s)"]
    - df_modelado_trainseguro["vgeo_esperada_train_km_s"]
)

columnas_residuos = [
    "residuo_lambda_train_deg",
    "residuo_beta_train_deg",
    "residuo_vgeo_train_km_s",
]

if not np.isfinite(
    df_modelado_trainseguro[columnas_residuos].to_numpy(dtype=float)
).all():
    raise ValueError("Los residuos del centro físico contienen valores no finitos.")


## 4. Velocidad residual y base física local


In [ ]:
# Construir los vectores de velocidad observados y esperados

def construir_vector_velocidad(lambda_deg, beta_deg, velocidad_km_s):
    lambda_rad = np.deg2rad(np.asarray(lambda_deg, dtype=float))
    beta_rad = np.deg2rad(np.asarray(beta_deg, dtype=float))
    velocidad_km_s = np.asarray(velocidad_km_s, dtype=float)

    componente_x = (
        velocidad_km_s
        * np.cos(beta_rad)
        * np.cos(lambda_rad)
    )

    componente_y = (
        velocidad_km_s
        * np.cos(beta_rad)
        * np.sin(lambda_rad)
    )

    componente_z = velocidad_km_s * np.sin(beta_rad)

    return np.column_stack(
        [componente_x, componente_y, componente_z]
    )


vectores_observados = construir_vector_velocidad(
    df_modelado_trainseguro["lambda_sce_deg"],
    df_modelado_trainseguro["BETgeo (deg)"],
    df_modelado_trainseguro["Vgeo (km/s)"],
)

vectores_esperados = construir_vector_velocidad(
    df_modelado_trainseguro["lambda_sce_esperada_train_deg"],
    df_modelado_trainseguro["beta_esperada_train_deg"],
    df_modelado_trainseguro["vgeo_esperada_train_km_s"],
)

vectores_residuales = vectores_observados - vectores_esperados

df_modelado_trainseguro[
    ["delta_v_x_km_s", "delta_v_y_km_s", "delta_v_z_km_s"]
] = vectores_residuales

df_modelado_trainseguro["norma_delta_v_km_s"] = np.linalg.norm(
    vectores_residuales,
    axis=1,
)

if not np.isfinite(vectores_residuales).all():
    raise ValueError("Los vectores residuales contienen valores no finitos.")


In [ ]:
# Construir la base local alrededor del radiante esperado y proyectar el residuo

def construir_base_local_velocidad(lambda_centro_deg, beta_centro_deg):
    lambda_rad = np.deg2rad(
        np.asarray(lambda_centro_deg, dtype=float)
    )

    beta_rad = np.deg2rad(
        np.asarray(beta_centro_deg, dtype=float)
    )

    eje_paralelo = np.column_stack(
        [
            np.cos(beta_rad) * np.cos(lambda_rad),
            np.cos(beta_rad) * np.sin(lambda_rad),
            np.sin(beta_rad),
        ]
    )

    eje_lambda = np.column_stack(
        [
            -np.sin(lambda_rad),
            np.cos(lambda_rad),
            np.zeros_like(lambda_rad),
        ]
    )

    eje_beta = np.column_stack(
        [
            -np.sin(beta_rad) * np.cos(lambda_rad),
            -np.sin(beta_rad) * np.sin(lambda_rad),
            np.cos(beta_rad),
        ]
    )

    return eje_paralelo, eje_lambda, eje_beta


ejes_paralelos, ejes_lambda, ejes_beta = (
    construir_base_local_velocidad(
        df_modelado_trainseguro["lambda_sce_esperada_train_deg"],
        df_modelado_trainseguro["beta_esperada_train_deg"],
    )
)

df_modelado_trainseguro["delta_v_paralela_km_s"] = np.einsum(
    "ij,ij->i",
    vectores_residuales,
    ejes_paralelos,
)

df_modelado_trainseguro["delta_v_lambda_km_s"] = np.einsum(
    "ij,ij->i",
    vectores_residuales,
    ejes_lambda,
)

df_modelado_trainseguro["delta_v_beta_km_s"] = np.einsum(
    "ij,ij->i",
    vectores_residuales,
    ejes_beta,
)


In [ ]:
# Comprobar las identidades geométricas de la descomposición local

componentes_locales = df_modelado_trainseguro[
    [
        "delta_v_paralela_km_s",
        "delta_v_lambda_km_s",
        "delta_v_beta_km_s",
    ]
].to_numpy(dtype=float)

vectores_reconstruidos = (
    componentes_locales[:, 0, None] * ejes_paralelos
    + componentes_locales[:, 1, None] * ejes_lambda
    + componentes_locales[:, 2, None] * ejes_beta
)

velocidades_observadas = df_modelado_trainseguro[
    "Vgeo (km/s)"
].to_numpy(dtype=float)

velocidades_esperadas = df_modelado_trainseguro[
    "vgeo_esperada_train_km_s"
].to_numpy(dtype=float)

direcciones_observadas = (
    vectores_observados
    / velocidades_observadas[:, None]
)

cosenos_angulares = np.clip(
    np.einsum(
        "ij,ij->i",
        direcciones_observadas,
        ejes_paralelos,
    ),
    -1.0,
    1.0,
)

paralela_teorica = (
    velocidades_observadas * cosenos_angulares
    - velocidades_esperadas
)

transversal_local = np.sqrt(
    componentes_locales[:, 1] ** 2
    + componentes_locales[:, 2] ** 2
)

transversal_teorica = (
    velocidades_observadas
    * np.sqrt(
        np.maximum(
            0.0,
            1.0 - cosenos_angulares ** 2,
        )
    )
)

filas_auditoria_local = [
    {
        "comprobacion": "Norma del eje paralelo",
        "error_maximo": np.max(
            np.abs(
                np.linalg.norm(ejes_paralelos, axis=1) - 1
            )
        ),
        "unidad": "sin unidad",
    },
    {
        "comprobacion": "Norma del eje lambda",
        "error_maximo": np.max(
            np.abs(
                np.linalg.norm(ejes_lambda, axis=1) - 1
            )
        ),
        "unidad": "sin unidad",
    },
    {
        "comprobacion": "Norma del eje beta",
        "error_maximo": np.max(
            np.abs(
                np.linalg.norm(ejes_beta, axis=1) - 1
            )
        ),
        "unidad": "sin unidad",
    },
    {
        "comprobacion": "Paralelo · lambda",
        "error_maximo": np.max(
            np.abs(
                np.einsum(
                    "ij,ij->i",
                    ejes_paralelos,
                    ejes_lambda,
                )
            )
        ),
        "unidad": "sin unidad",
    },
    {
        "comprobacion": "Paralelo · beta",
        "error_maximo": np.max(
            np.abs(
                np.einsum(
                    "ij,ij->i",
                    ejes_paralelos,
                    ejes_beta,
                )
            )
        ),
        "unidad": "sin unidad",
    },
    {
        "comprobacion": "Lambda · beta",
        "error_maximo": np.max(
            np.abs(
                np.einsum(
                    "ij,ij->i",
                    ejes_lambda,
                    ejes_beta,
                )
            )
        ),
        "unidad": "sin unidad",
    },
    {
        "comprobacion": "Orientación lambda × beta = paralelo",
        "error_maximo": np.max(
            np.linalg.norm(
                np.cross(ejes_lambda, ejes_beta)
                - ejes_paralelos,
                axis=1,
            )
        ),
        "unidad": "sin unidad",
    },
    {
        "comprobacion": "Reconstrucción del vector residual",
        "error_maximo": np.max(
            np.linalg.norm(
                vectores_reconstruidos
                - vectores_residuales,
                axis=1,
            )
        ),
        "unidad": "km/s",
    },
    {
        "comprobacion": "Conservación de la norma residual",
        "error_maximo": np.max(
            np.abs(
                np.linalg.norm(componentes_locales, axis=1)
                - df_modelado_trainseguro[
                    "norma_delta_v_km_s"
                ].to_numpy(dtype=float)
            )
        ),
        "unidad": "km/s",
    },
    {
        "comprobacion": "Identidad de la componente paralela",
        "error_maximo": np.max(
            np.abs(
                componentes_locales[:, 0]
                - paralela_teorica
            )
        ),
        "unidad": "km/s",
    },
    {
        "comprobacion": "Identidad de la componente transversal",
        "error_maximo": np.max(
            np.abs(
                transversal_local
                - transversal_teorica
            )
        ),
        "unidad": "km/s",
    },
]

df_auditoria_componentes_locales = pd.DataFrame(
    filas_auditoria_local
)

if not np.isfinite(componentes_locales).all():
    raise ValueError(
        "Las componentes locales contienen valores no finitos."
    )

if (
    df_auditoria_componentes_locales["error_maximo"].max()
    > 1e-9
):
    raise ValueError(
        "La descomposición local no superó la tolerancia numérica."
    )

display(df_auditoria_componentes_locales)


## 5. Controles observacionales y clases de brillo


In [ ]:
# Contar las estaciones participantes de cada trayectoria

def contar_estaciones(valor):
    if pd.isna(valor):
        return pd.NA

    estaciones = [
        estacion.strip()
        for estacion in str(valor).split(",")
        if estacion.strip()
    ]

    return len(set(estaciones))


df_modelado_trainseguro["numero_estaciones"] = (
    df_modelado_trainseguro["Participating (stations)"]
    .map(contar_estaciones)
    .astype("Int64")
)

if df_modelado_trainseguro["numero_estaciones"].isna().any():
    raise ValueError(
        "Hay trayectorias sin número de estaciones disponible."
    )

if (df_modelado_trainseguro["numero_estaciones"] < 2).any():
    raise ValueError(
        "Se encontraron trayectorias con menos de dos estaciones."
    )


In [ ]:
# Preparar la incertidumbre de velocidad usando únicamente entrenamiento

mascara_entrenamiento = (
    df_modelado_trainseguro["particion_modelado"]
    == "entrenamiento"
)

mascara_vgeo_sigma_cero = (
    df_modelado_trainseguro["Vgeo_sigma"] == 0
)

bandera_guardada = (
    df_modelado_trainseguro["flag_vgeo_sigma_cero"]
    .astype(bool)
)

if not (
    mascara_vgeo_sigma_cero
    == bandera_guardada
).all():
    raise ValueError(
        "La bandera de Vgeo_sigma igual a cero "
        "no coincide con los valores observados."
    )

mediana_vgeo_sigma_train = (
    df_modelado_trainseguro.loc[
        mascara_entrenamiento
        & ~mascara_vgeo_sigma_cero,
        "Vgeo_sigma",
    ]
    .median()
)

if not np.isfinite(mediana_vgeo_sigma_train):
    raise ValueError(
        "No se pudo calcular la sustitución de Vgeo_sigma."
    )

df_modelado_trainseguro["Vgeo_sigma_modelo"] = (
    df_modelado_trainseguro["Vgeo_sigma"]
    .mask(
        mascara_vgeo_sigma_cero,
        mediana_vgeo_sigma_train,
    )
)

columnas_calidad_modelo = [
    "Qc (deg)",
    "MedianFitErr (arcsec)",
    "numero_estaciones",
    "sigma_radiante_aprox",
    "Vgeo_sigma_modelo",
]

if not np.isfinite(
    df_modelado_trainseguro[
        columnas_calidad_modelo
    ].to_numpy(dtype=float)
).all():
    raise ValueError(
        "Los controles observacionales contienen valores no finitos."
    )

print(
    "Mediana positiva de Vgeo_sigma aprendida con entrenamiento:",
    mediana_vgeo_sigma_train,
)


In [ ]:
# Definir las clases de brillo con terciles calculados exclusivamente en entrenamiento

mascara_fotometria_train = (
    mascara_entrenamiento
    & df_modelado_trainseguro["fotometria_utilizable"]
    & df_modelado_trainseguro["magnitud_analisis"].notna()
)

magnitudes_train = df_modelado_trainseguro.loc[
    mascara_fotometria_train,
    "magnitud_analisis",
]

limite_brillante_train = magnitudes_train.quantile(1 / 3)
limite_tenue_train = magnitudes_train.quantile(2 / 3)

mascara_magnitud_valida = (
    df_modelado_trainseguro["fotometria_utilizable"]
    & df_modelado_trainseguro["magnitud_analisis"].notna()
)

mascara_brillante = (
    mascara_magnitud_valida
    & (
        df_modelado_trainseguro["magnitud_analisis"]
        <= limite_brillante_train
    )
)

mascara_tenue = (
    mascara_magnitud_valida
    & (
        df_modelado_trainseguro["magnitud_analisis"]
        >= limite_tenue_train
    )
)

mascara_intermedia = (
    mascara_magnitud_valida
    & ~mascara_brillante
    & ~mascara_tenue
)

df_modelado_trainseguro["grupo_brillo_ml"] = pd.Series(
    pd.NA,
    index=df_modelado_trainseguro.index,
    dtype="string",
)

df_modelado_trainseguro.loc[
    mascara_brillante,
    "grupo_brillo_ml",
] = "brillante"

df_modelado_trainseguro.loc[
    mascara_intermedia,
    "grupo_brillo_ml",
] = "intermedio"

df_modelado_trainseguro.loc[
    mascara_tenue,
    "grupo_brillo_ml",
] = "tenue"

df_modelado_trainseguro["objetivo_ml"] = pd.Series(
    pd.NA,
    index=df_modelado_trainseguro.index,
    dtype="Int64",
)

df_modelado_trainseguro.loc[
    mascara_brillante,
    "objetivo_ml",
] = 0

df_modelado_trainseguro.loc[
    mascara_tenue,
    "objetivo_ml",
] = 1

df_modelado_trainseguro["usar_modelo_ml"] = (
    df_modelado_trainseguro["objetivo_ml"].notna()
)

# El resumen mostrado aquí utiliza solo desarrollo.
conteos_train = (
    df_modelado_trainseguro.loc[
        mascara_entrenamiento,
        "grupo_brillo_ml",
    ]
    .value_counts(dropna=False)
)

resumen_grupos_entrenamiento = pd.DataFrame(
    [
        {
            "brillantes": int(conteos_train.get("brillante", 0)),
            "intermedias": int(conteos_train.get("intermedio", 0)),
            "tenues": int(conteos_train.get("tenue", 0)),
            "sin_fotometria_utilizable": int(
                df_modelado_trainseguro.loc[
                    mascara_entrenamiento,
                    "grupo_brillo_ml",
                ].isna().sum()
            ),
            "utilizables_ml": int(
                df_modelado_trainseguro.loc[
                    mascara_entrenamiento,
                    "usar_modelo_ml",
                ].sum()
            ),
        }
    ]
)

print("Límite brillante:", limite_brillante_train)
print("Límite tenue:", limite_tenue_train)
display(resumen_grupos_entrenamiento)


## 6. Base final de modelado y documentación


In [ ]:
# Construir las bases finales utilizadas por los notebooks 05–07

columnas_pca = [
    "delta_v_paralela_km_s",
    "delta_v_lambda_km_s",
    "delta_v_beta_km_s",
]

df_modelado_trainseguro["usar_pca"] = np.isfinite(
    df_modelado_trainseguro[columnas_pca]
).all(axis=1)

columnas_base_modelado = [
    "id_fila_origen",
    "Beginning (UTC Time)",
    "source_year",
    "particion_modelado",
    "pertenece_muestra_estricta",
    "nivel_seleccion",
    "fotometria_utilizable",
    "Peak (AbsMag)",
    "magnitud_analisis",
    "flag_fotometria_sospechosa",
    "grupo_brillo_ml",
    "objetivo_ml",
    "usar_modelo_ml",
    "usar_pca",
    "delta_sol_lon_deg",
    "lambda_sce_deg",
    "BETgeo (deg)",
    "Vgeo (km/s)",
    "lambda_sce_esperada_train_deg",
    "beta_esperada_train_deg",
    "vgeo_esperada_train_km_s",
    "residuo_lambda_train_deg",
    "residuo_beta_train_deg",
    "residuo_vgeo_train_km_s",
    "delta_v_x_km_s",
    "delta_v_y_km_s",
    "delta_v_z_km_s",
    "delta_v_paralela_km_s",
    "delta_v_lambda_km_s",
    "delta_v_beta_km_s",
    "norma_delta_v_km_s",
    "Qc (deg)",
    "MedianFitErr (arcsec)",
    "Participating (stations)",
    "numero_estaciones",
    "sigma_radiante_aprox",
    "Vgeo_sigma",
    "flag_vgeo_sigma_cero",
    "Vgeo_sigma_modelo",
]

df_base_modelado_final = (
    df_modelado_trainseguro[
        columnas_base_modelado
    ]
    .copy()
)

df_modelado_entrenamiento = (
    df_base_modelado_final.loc[
        df_base_modelado_final[
            "particion_modelado"
        ]
        == "entrenamiento"
    ]
    .copy()
)

df_modelado_prueba = (
    df_base_modelado_final.loc[
        df_base_modelado_final[
            "particion_modelado"
        ]
        == "prueba"
    ]
    .copy()
)

if not df_modelado_entrenamiento["id_fila_origen"].is_unique:
    raise ValueError("Hay IDs repetidos en entrenamiento.")

if not df_modelado_prueba["id_fila_origen"].is_unique:
    raise ValueError("Hay IDs repetidos en prueba.")

if not df_base_modelado_final["usar_pca"].all():
    raise ValueError(
        "Hay trayectorias sin componentes locales finitas."
    )

if df_base_modelado_final[
    columnas_calidad_modelo
].isna().any().any():
    raise ValueError(
        "Hay faltantes en los controles observacionales."
    )

df_auditoria_base_modelado = pd.DataFrame(
    [
        {
            "comprobacion": "Filas totales",
            "resultado": len(df_base_modelado_final),
        },
        {
            "comprobacion": "Filas de entrenamiento",
            "resultado": len(df_modelado_entrenamiento),
        },
        {
            "comprobacion": "Filas de prueba",
            "resultado": len(df_modelado_prueba),
        },
        {
            "comprobacion": "Variables finales",
            "resultado": len(columnas_base_modelado),
        },
        {
            "comprobacion": "IDs únicos entrenamiento",
            "resultado": df_modelado_entrenamiento[
                "id_fila_origen"
            ].is_unique,
        },
        {
            "comprobacion": "IDs únicos prueba",
            "resultado": df_modelado_prueba[
                "id_fila_origen"
            ].is_unique,
        },
        {
            "comprobacion": "Filas PCA entrenamiento",
            "resultado": int(
                df_modelado_entrenamiento[
                    "usar_pca"
                ].sum()
            ),
        },
    ]
)

display(df_auditoria_base_modelado)


In [ ]:
# Extender el diccionario físico con las componentes locales y las demás variables de modelado

documentacion_nueva = {
    "particion_modelado": {"categoria": "Partición", "unidad": "categoría", "descripcion": "Indica si la trayectoria pertenece al entrenamiento de 2019–2024 o a la prueba temporal independiente de 2025."},
    "grupo_brillo_ml": {"categoria": "Objetivo ML", "unidad": "categoría", "descripcion": "Grupo de brillo definido con los terciles de magnitud del entrenamiento: brillante, intermedio o tenue; queda vacío cuando la fotometría no es utilizable."},
    "objetivo_ml": {"categoria": "Objetivo ML", "unidad": "clase binaria", "descripcion": "Variable objetivo del clasificador: 0 para meteoro brillante y 1 para meteoro tenue; queda vacía para casos intermedios o sin fotometría utilizable."},
    "usar_modelo_ml": {"categoria": "Objetivo ML", "unidad": "booleano", "descripcion": "Indica si la trayectoria pertenece a uno de los dos grupos extremos de brillo y puede utilizarse en la clasificación binaria."},
    "usar_pca": {"categoria": "PCA", "unidad": "booleano", "descripcion": "Indica si las componentes paralela, longitudinal y latitudinal de la velocidad residual están disponibles y son finitas para el PCA."},
    "lambda_sce_esperada_train_deg": {"categoria": "Centro físico de entrenamiento", "unidad": "grados", "descripcion": "Longitud eclíptica centrada en el Sol esperada según el modelo lineal ajustado exclusivamente con 2019–2024."},
    "beta_esperada_train_deg": {"categoria": "Centro físico de entrenamiento", "unidad": "grados", "descripcion": "Latitud eclíptica esperada según el modelo cuadrático ajustado exclusivamente con 2019–2024."},
    "vgeo_esperada_train_km_s": {"categoria": "Centro físico de entrenamiento", "unidad": "km/s", "descripcion": "Velocidad geocéntrica esperada según el modelo lineal ajustado exclusivamente con 2019–2024."},
    "residuo_lambda_train_deg": {"categoria": "Residuo físico", "unidad": "grados", "descripcion": "Diferencia angular envuelta entre la longitud SCE observada y la esperada por el modelo de entrenamiento."},
    "residuo_beta_train_deg": {"categoria": "Residuo físico", "unidad": "grados", "descripcion": "Diferencia entre la latitud eclíptica observada y la esperada por el modelo de entrenamiento."},
    "residuo_vgeo_train_km_s": {"categoria": "Residuo físico", "unidad": "km/s", "descripcion": "Diferencia entre la velocidad geocéntrica observada y la esperada por el modelo de entrenamiento."},
    "delta_v_x_km_s": {"categoria": "Velocidad residual", "unidad": "km/s", "descripcion": "Componente x del vector de velocidad observado menos el esperado, expresado en los ejes cartesianos del sistema eclíptico centrado en el Sol."},
    "delta_v_y_km_s": {"categoria": "Velocidad residual", "unidad": "km/s", "descripcion": "Componente y del vector de velocidad observado menos el esperado, expresado en los ejes cartesianos del sistema eclíptico centrado en el Sol."},
    "delta_v_z_km_s": {"categoria": "Velocidad residual", "unidad": "km/s", "descripcion": "Componente z del vector de velocidad observado menos el esperado, expresado en los ejes cartesianos del sistema eclíptico centrado en el Sol."},
    "delta_v_paralela_km_s": {"categoria": "PCA", "unidad": "km/s", "descripcion": "Proyección del vector de velocidad residual sobre la dirección del radiante esperado; un valor positivo indica una componente observada mayor en esa dirección."},
    "delta_v_lambda_km_s": {"categoria": "PCA", "unidad": "km/s", "descripcion": "Proyección del vector de velocidad residual sobre la dirección tangente local de longitud eclíptica centrada en el Sol creciente."},
    "delta_v_beta_km_s": {"categoria": "PCA", "unidad": "km/s", "descripcion": "Proyección del vector de velocidad residual sobre la dirección tangente local de latitud eclíptica creciente."},
    "norma_delta_v_km_s": {"categoria": "Velocidad residual", "unidad": "km/s", "descripcion": "Norma euclidiana del vector tridimensional de velocidad observado menos el esperado; es invariante ante el cambio entre bases cartesianas y locales."},
    "numero_estaciones": {"categoria": "Calidad observacional", "unidad": "conteo", "descripcion": "Número de estaciones participantes distintas registradas para la trayectoria."},
    "Vgeo_sigma_modelo": {"categoria": "Calidad observacional", "unidad": "km/s", "descripcion": "Incertidumbre de velocidad utilizada por el modelo; los valores originales iguales a cero fueron sustituidos por la mediana positiva del entrenamiento."}
}

variables_documentadas_originales = set(df_diccionario_fisico["variable"])
variables_documentadas_nuevas = set(documentacion_nueva)
variables_sin_documentacion = set(columnas_base_modelado) - variables_documentadas_originales - variables_documentadas_nuevas

if variables_sin_documentacion:
    raise ValueError(f"Faltan descripciones para: {sorted(variables_sin_documentacion)}")

df_diccionario_modelado = df_diccionario_fisico.loc[df_diccionario_fisico["variable"].isin(columnas_base_modelado)].copy()
filas_documentacion_nueva = []

for variable, documentacion in documentacion_nueva.items():
    serie = df_base_modelado_final[variable]

    filas_documentacion_nueva.append({
        "variable": variable,
        "categoria": documentacion["categoria"],
        "tipo_dato": str(serie.dtype),
        "unidad": documentacion["unidad"],
        "descripcion": documentacion["descripcion"],
        "valores_faltantes": int(serie.isna().sum()),
        "valores_unicos": int(serie.nunique(dropna=True))
    })

df_diccionario_modelado = pd.concat([df_diccionario_modelado, pd.DataFrame(filas_documentacion_nueva)], ignore_index=True)
orden_columnas_modelado = {variable: posicion for posicion, variable in enumerate(columnas_base_modelado, start=1)}
df_diccionario_modelado["posicion"] = df_diccionario_modelado["variable"].map(orden_columnas_modelado)
df_diccionario_modelado = df_diccionario_modelado.sort_values("posicion").reset_index(drop=True)
df_diccionario_modelado = df_diccionario_modelado[["posicion", "variable", "categoria", "tipo_dato", "unidad", "descripcion", "valores_faltantes", "valores_unicos"]]

if len(df_diccionario_modelado) != len(columnas_base_modelado):
    raise ValueError("El diccionario final no contiene exactamente una fila por variable.")

display(df_diccionario_modelado)
print("Variables de modelado:", len(columnas_base_modelado))
print("Variables documentadas:", len(df_diccionario_modelado))


In [ ]:
# Guardar la configuración metodológica del producto v2

df_configuracion_modelado = pd.DataFrame(
    [
        {
            "elemento": "Versión",
            "valor": "geminidas_modelado_v2",
            "descripcion": "Base de modelado con velocidad residual expresada en componentes físicas locales.",
        },
        {
            "elemento": "Años de entrenamiento",
            "valor": "2019–2024",
            "descripcion": "Años utilizados para ajustar la referencia física y para desarrollar PCA y modelos.",
        },
        {
            "elemento": "Año de prueba",
            "valor": "2025",
            "descripcion": "Partición reservada para la evaluación predictiva posterior.",
        },
        {
            "elemento": "Modelo de longitud",
            "valor": "Lineal común",
            "descripcion": "lambda_sce_deg en función de delta_sol_lon_deg, ajustado solo con 2019–2024.",
        },
        {
            "elemento": "Modelo de latitud",
            "valor": "Cuadrático común",
            "descripcion": "BETgeo (deg) en función de delta_sol_lon_deg, ajustado solo con 2019–2024.",
        },
        {
            "elemento": "Modelo de velocidad",
            "valor": "Lineal común",
            "descripcion": "Vgeo (km/s) en función de delta_sol_lon_deg, ajustado solo con 2019–2024.",
        },
        {
            "elemento": "Convención del vector residual",
            "valor": "velocidad observada menos velocidad esperada",
            "descripcion": "delta_v = v_observada - v_esperada, usando vectores orientados hacia el radiante.",
        },
        {
            "elemento": "Base física local",
            "valor": "paralela, lambda, beta",
            "descripcion": "Proyección del residuo sobre la dirección esperada y las dos direcciones tangentes locales.",
        },
        {
            "elemento": "Variables del PCA",
            "valor": ", ".join(columnas_pca),
            "descripcion": "Tres componentes locales de la velocidad residual, todas en km/s.",
        },
        {
            "elemento": "Ajuste del PCA",
            "valor": "Solo entrenamiento",
            "descripcion": "El PCA debe aprenderse con 2019–2024 y aplicarse sin reajuste a 2025.",
        },
        {
            "elemento": "Escalamiento del PCA",
            "valor": "Centrar sin estandarizar",
            "descripcion": "Se conserva la varianza física relativa de las tres componentes en km/s.",
        },
        {
            "elemento": "Límite brillante",
            "valor": limite_brillante_train,
            "descripcion": "Tercil inferior de magnitud calculado únicamente con entrenamiento.",
        },
        {
            "elemento": "Límite tenue",
            "valor": limite_tenue_train,
            "descripcion": "Tercil superior de magnitud calculado únicamente con entrenamiento.",
        },
        {
            "elemento": "Codificación del objetivo",
            "valor": "0 = brillante; 1 = tenue",
            "descripcion": "Los casos intermedios y sin fotometría utilizable no entran al clasificador binario.",
        },
        {
            "elemento": "Controles de calidad",
            "valor": ", ".join(columnas_calidad_modelo),
            "descripcion": "Controles observacionales disponibles para el modelo de referencia.",
        },
        {
            "elemento": "Tratamiento de Vgeo_sigma igual a cero",
            "valor": mediana_vgeo_sigma_train,
            "descripcion": "Sustitución por la mediana positiva aprendida exclusivamente con entrenamiento.",
        },
        {
            "elemento": "Muestra estándar",
            "valor": "Análisis principal",
            "descripcion": "Muestra principal de la investigación.",
        },
        {
            "elemento": "Muestra estricta",
            "valor": "Análisis de sensibilidad",
            "descripcion": "Subconjunto de mayor calidad identificado por pertenece_muestra_estricta.",
        },
    ]
)

df_resumen_particiones_modelado = pd.DataFrame(
    [
        {
            "particion": "entrenamiento",
            "anios": "2019–2024",
            "trayectorias": len(df_modelado_entrenamiento),
            "utilizables_pca": int(
                df_modelado_entrenamiento["usar_pca"].sum()
            ),
            "trayectorias_estrictas": int(
                df_modelado_entrenamiento[
                    "pertenece_muestra_estricta"
                ].sum()
            ),
        },
        {
            "particion": "prueba",
            "anios": "2025",
            "trayectorias": len(df_modelado_prueba),
            "utilizables_pca": int(
                df_modelado_prueba["usar_pca"].sum()
            ),
            "trayectorias_estrictas": int(
                df_modelado_prueba[
                    "pertenece_muestra_estricta"
                ].sum()
            ),
        },
    ]
)

display(df_configuracion_modelado)
display(df_resumen_particiones_modelado)


## 7. Guardado y verificación


In [ ]:
# Guardar únicamente las dos particiones utilizadas por los notebooks posteriores

carpeta_metadata_modelado = (
    carpeta_modelado / "metadata"
)

carpeta_modelado.mkdir(
    parents=True,
    exist_ok=True,
)

carpeta_metadata_modelado.mkdir(
    parents=True,
    exist_ok=True,
)

ruta_entrenamiento_parquet = (
    carpeta_modelado
    / "geminidas_modelado_entrenamiento_2019_2024_v2.parquet"
)

ruta_prueba_parquet = (
    carpeta_modelado
    / "geminidas_modelado_prueba_2025_v2.parquet"
)

ruta_diccionario_modelado = (
    carpeta_metadata_modelado
    / "diccionario_variables_modelado_v2.csv"
)

ruta_configuracion_modelado = (
    carpeta_metadata_modelado
    / "configuracion_modelado_v2.csv"
)

ruta_parametros_centro = (
    carpeta_metadata_modelado
    / "parametros_centro_entrenamiento_v2.csv"
)

ruta_resumen_particiones = (
    carpeta_metadata_modelado
    / "resumen_particiones_modelado_v2.csv"
)

ruta_auditoria_base = (
    carpeta_metadata_modelado
    / "auditoria_base_modelado_v2.csv"
)

ruta_auditoria_componentes_locales = (
    carpeta_metadata_modelado
    / "auditoria_componentes_locales_v2.csv"
)

df_modelado_entrenamiento.to_parquet(
    ruta_entrenamiento_parquet,
    index=False,
)

df_modelado_prueba.to_parquet(
    ruta_prueba_parquet,
    index=False,
)

df_diccionario_modelado.to_csv(
    ruta_diccionario_modelado,
    index=False,
)

df_configuracion_modelado.to_csv(
    ruta_configuracion_modelado,
    index=False,
)

df_parametros_centro_train.to_csv(
    ruta_parametros_centro,
    index=False,
)

df_resumen_particiones_modelado.to_csv(
    ruta_resumen_particiones,
    index=False,
)

df_auditoria_base_modelado.to_csv(
    ruta_auditoria_base,
    index=False,
)

df_auditoria_componentes_locales.to_csv(
    ruta_auditoria_componentes_locales,
    index=False,
)

rutas_generadas_modelado = [
    ruta_entrenamiento_parquet,
    ruta_prueba_parquet,
    ruta_diccionario_modelado,
    ruta_configuracion_modelado,
    ruta_parametros_centro,
    ruta_resumen_particiones,
    ruta_auditoria_base,
    ruta_auditoria_componentes_locales,
]

for ruta in rutas_generadas_modelado:
    print(ruta.relative_to(raiz_proyecto))


In [ ]:
# Recargar los productos principales y generar el manifiesto de integridad

def normalizar_texto(serie):
    return (
        serie.astype("string")
        .fillna("<NA>")
        .to_numpy(dtype=str)
    )


def verificar_base_guardada(nombre, df_leido, df_original):
    objetivos_leidos = pd.to_numeric(
        df_leido["objetivo_ml"],
        errors="coerce",
    ).to_numpy(dtype=float)

    objetivos_originales = pd.to_numeric(
        df_original["objetivo_ml"],
        errors="coerce",
    ).to_numpy(dtype=float)

    return {
        "archivo": nombre,
        "filas_correctas": len(df_leido) == len(df_original),
        "columnas_correctas": (
            df_leido.columns.tolist()
            == df_original.columns.tolist()
        ),
        "identificadores_correctos": np.array_equal(
            df_leido["id_fila_origen"].to_numpy(),
            df_original["id_fila_origen"].to_numpy(),
        ),
        "particiones_correctas": np.array_equal(
            normalizar_texto(
                df_leido["particion_modelado"]
            ),
            normalizar_texto(
                df_original["particion_modelado"]
            ),
        ),
        "grupos_brillo_correctos": np.array_equal(
            normalizar_texto(
                df_leido["grupo_brillo_ml"]
            ),
            normalizar_texto(
                df_original["grupo_brillo_ml"]
            ),
        ),
        "objetivos_correctos": np.allclose(
            objetivos_leidos,
            objetivos_originales,
            equal_nan=True,
        ),
        "error_maximo_componentes_locales": np.max(
            np.abs(
                df_leido[columnas_pca].to_numpy(dtype=float)
                - df_original[columnas_pca].to_numpy(dtype=float)
            )
        ),
        "error_maximo_norma": np.max(
            np.abs(
                df_leido["norma_delta_v_km_s"].to_numpy(dtype=float)
                - df_original["norma_delta_v_km_s"].to_numpy(dtype=float)
            )
        ),
    }


df_entrenamiento_recargado = pd.read_parquet(
    ruta_entrenamiento_parquet
)

df_prueba_recargado = pd.read_parquet(
    ruta_prueba_parquet
)

df_verificacion_guardado = pd.DataFrame(
    [
        verificar_base_guardada(
            "Entrenamiento",
            df_entrenamiento_recargado,
            df_modelado_entrenamiento,
        ),
        verificar_base_guardada(
            "Prueba",
            df_prueba_recargado,
            df_modelado_prueba,
        ),
    ]
)

columnas_booleanas = [
    "filas_correctas",
    "columnas_correctas",
    "identificadores_correctos",
    "particiones_correctas",
    "grupos_brillo_correctos",
    "objetivos_correctos",
]

if not df_verificacion_guardado[
    columnas_booleanas
].to_numpy().all():
    raise ValueError(
        "Alguna base guardada no coincide con su versión en memoria."
    )

errores_numericos = df_verificacion_guardado[
    [
        "error_maximo_componentes_locales",
        "error_maximo_norma",
    ]
].to_numpy(dtype=float)

if errores_numericos.max() > 1e-12:
    raise ValueError(
        "La escritura introdujo diferencias numéricas mayores que la tolerancia."
    )

ruta_verificacion_guardado = (
    carpeta_metadata_modelado
    / "verificacion_guardado_modelado_v2.csv"
)

df_verificacion_guardado.to_csv(
    ruta_verificacion_guardado,
    index=False,
)


def calcular_sha256(ruta):
    sha256 = hashlib.sha256()

    with ruta.open("rb") as archivo:
        for bloque in iter(
            lambda: archivo.read(1024 * 1024),
            b"",
        ):
            sha256.update(bloque)

    return sha256.hexdigest()


rutas_para_manifiesto = [
    *rutas_generadas_modelado,
    ruta_verificacion_guardado,
]

df_manifiesto_integridad = pd.DataFrame(
    [
        {
            "archivo": str(
                ruta.relative_to(carpeta_modelado)
            ),
            "tamano_bytes": ruta.stat().st_size,
            "sha256": calcular_sha256(ruta),
        }
        for ruta in rutas_para_manifiesto
    ]
)

ruta_manifiesto_integridad = (
    carpeta_metadata_modelado
    / "manifiesto_integridad_modelado_v2.csv"
)

df_manifiesto_integridad.to_csv(
    ruta_manifiesto_integridad,
    index=False,
)

display(df_verificacion_guardado)

print(
    "Archivos incluidos en el manifiesto:",
    len(df_manifiesto_integridad),
)

print(
    "Manifiesto:",
    ruta_manifiesto_integridad.relative_to(
        raiz_proyecto
    ),
)
